# Clase 03: Estabilidad y regularizacion

Entrenar redes profundas no consiste solo en apilar capas. En cuanto aumenta la profundidad aparecen preguntas estructurales: como inicializar, como evitar gradientes pobres, cuando conviene normalizar y por que AdamW desplazo a variantes mas viejas para muchos pipelines practicos.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.datasets import make_classification
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

MODULE_ROOT = Path.cwd().resolve().parent
if str(MODULE_ROOT) not in sys.path:
    sys.path.append(str(MODULE_ROOT))

from tools.notebook_utils import choose_value, configure_runtime, plot_history

runtime = configure_runtime(seed=19)
print(runtime.summary())
        


## Por que existe esta clase

Cuando una red crece, aparecen dos problemas muy concretos:

- el gradiente puede desvanecerse o explotar,
- el modelo puede memorizar con facilidad y generalizar mal.

La respuesta moderna no es una sola tecnica, sino una combinacion de buenas practicas: inicializacion acorde a la activacion, normalizacion, regularizacion, optimizadores robustos y criterio para frenar el entrenamiento.


In [ ]:
X, y = make_classification(
    n_samples=int(choose_value(2600, 900)),
    n_features=20,
    n_informative=12,
    n_redundant=4,
    n_clusters_per_class=2,
    class_sep=1.1,
    flip_y=0.03,
    random_state=19,
)

X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=19, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=19, stratify=y_temp)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_ds = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test, dtype=torch.long))

train_loader = DataLoader(train_ds, batch_size=int(choose_value(64, 32)), shuffle=True)
val_loader = DataLoader(val_ds, batch_size=128)
test_loader = DataLoader(test_ds, batch_size=128)
        


In [ ]:
class DeepMLP(torch.nn.Module):
    def __init__(self, input_dim: int, hidden_dims: list[int], norm: str | None = None, dropout: float = 0.0) -> None:
        super().__init__()
        layers: list[torch.nn.Module] = []
        current_dim = input_dim
        for hidden_dim in hidden_dims:
            layers.append(torch.nn.Linear(current_dim, hidden_dim))
            if norm == 'batch':
                layers.append(torch.nn.BatchNorm1d(hidden_dim))
            elif norm == 'layer':
                layers.append(torch.nn.LayerNorm(hidden_dim))
            layers.append(torch.nn.ReLU())
            if dropout > 0:
                layers.append(torch.nn.Dropout(dropout))
            current_dim = hidden_dim
        layers.append(torch.nn.Linear(current_dim, 2))
        self.model = torch.nn.Sequential(*layers)

    def forward(self, features: torch.Tensor) -> torch.Tensor:
        return self.model(features)


def initialize_model(model: torch.nn.Module, mode: str) -> None:
    for module in model.modules():
        if isinstance(module, torch.nn.Linear):
            if mode == 'kaiming':
                torch.nn.init.kaiming_normal_(module.weight)
            elif mode == 'xavier':
                torch.nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)


def first_backward_snapshot(model: torch.nn.Module) -> float:
    model = model.to(runtime.device)
    model.train()
    features, targets = next(iter(train_loader))
    features = features.to(runtime.device)
    targets = targets.to(runtime.device)
    logits = model(features)
    loss = torch.nn.functional.cross_entropy(logits, targets)
    loss.backward()
    first_linear = next(module for module in model.modules() if isinstance(module, torch.nn.Linear))
    grad_norm = float(first_linear.weight.grad.norm().item())
    model.zero_grad(set_to_none=True)
    return grad_norm

bad_init_model = DeepMLP(input_dim=20, hidden_dims=[128, 128, 128, 128], norm=None, dropout=0.0)
good_init_model = DeepMLP(input_dim=20, hidden_dims=[128, 128, 128, 128], norm='layer', dropout=0.1)
initialize_model(good_init_model, 'kaiming')

print({
    'grad_norm_default_init': round(first_backward_snapshot(bad_init_model), 5),
    'grad_norm_kaiming_layer_norm': round(first_backward_snapshot(good_init_model), 5),
})
        


## Interpretacion rapida

- Una inicializacion mala puede dejar la senal demasiado debil o demasiado grande.
- `BatchNorm` ayuda cuando el batch es razonable y el problema es tabular o visual clasico.
- `LayerNorm` es mas estable cuando la dimension del batch cambia y es la normalizacion dominante en Transformers.
- `AdamW` separa el `weight_decay` del gradiente adaptativo, por eso es la opcion moderna mas comun.


In [ ]:
loss_fn = torch.nn.CrossEntropyLoss()
epochs = int(choose_value(28, 10))
patience = int(choose_value(6, 3))


def run_variant(name: str, norm: str | None, dropout: float, init: str | None, lr: float, weight_decay: float) -> dict:
    model = DeepMLP(input_dim=20, hidden_dims=[128, 128, 128, 128], norm=norm, dropout=dropout).to(runtime.device)
    if init is not None:
        initialize_model(model, init)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=max(epochs // 3, 1), gamma=0.6)

    history = {'train_loss': [], 'val_loss': [], 'val_acc': []}
    best_state = None
    best_val_loss = float('inf')
    patience_left = patience

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for features, targets in train_loader:
            features = features.to(runtime.device)
            targets = targets.to(runtime.device)
            optimizer.zero_grad()
            logits = model(features)
            loss = loss_fn(logits, targets)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(features)

        model.eval()
        val_loss_total = 0.0
        predictions = []
        targets_all = []
        with torch.inference_mode():
            for features, targets in val_loader:
                features = features.to(runtime.device)
                targets = targets.to(runtime.device)
                logits = model(features)
                val_loss_total += loss_fn(logits, targets).item() * len(features)
                predictions.extend(logits.argmax(dim=1).cpu().tolist())
                targets_all.extend(targets.cpu().tolist())

        train_loss = total_loss / len(train_loader.dataset)
        val_loss = val_loss_total / len(val_loader.dataset)
        val_acc = accuracy_score(targets_all, predictions)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left == 0:
                break

        scheduler.step()

    if best_state is not None:
        model.load_state_dict(best_state)

    model.eval()
    test_predictions = []
    test_targets = []
    with torch.inference_mode():
        for features, targets in test_loader:
            features = features.to(runtime.device)
            logits = model(features)
            test_predictions.extend(logits.argmax(dim=1).cpu().tolist())
            test_targets.extend(targets.tolist())

    return {
        'name': name,
        'history': history,
        'test_acc': accuracy_score(test_targets, test_predictions),
        'epochs_ran': len(history['val_loss']),
    }


results = [
    run_variant('baseline_profunda', norm=None, dropout=0.0, init=None, lr=0.003, weight_decay=0.0),
    run_variant('batch_norm_adamw', norm='batch', dropout=0.0, init='kaiming', lr=0.003, weight_decay=1e-3),
    run_variant('layer_norm_dropout', norm='layer', dropout=0.15, init='kaiming', lr=0.003, weight_decay=2e-3),
]

for result in results:
    print({
        'variant': result['name'],
        'test_acc': round(result['test_acc'], 3),
        'epochs_ran': result['epochs_ran'],
    })
        


In [ ]:
plt.figure(figsize=(8, 4))
for result in results:
    plt.plot(result['history']['val_acc'], label=result['name'])
plt.title('Comparacion de variantes: val_acc')
plt.xlabel('Epoca')
plt.ylabel('Accuracy')
plt.legend()
plt.tight_layout()

stable_result = max(results, key=lambda item: item['test_acc'])
plot_history(stable_result['history'], metrics=('train_loss', 'val_loss'), title=f"Mejor variante: {stable_result['name']}")
        


## Para cerrar

### Lo que deberia quedarte claro

- entrenar profundo exige estabilidad numerica, no solo mas compute,
- la eleccion de la inicializacion depende de la activacion,
- regularizar no es castigar al modelo: es empujarlo a generalizar,
- early stopping existe para frenar un proceso que ya no mejora lo importante.

### Ideas para experimentar

- sube el dropout y observa si empiezas a subajustar,
- cambia `BatchNorm` por `LayerNorm` y compara curva y costo,
- reemplaza `AdamW` por `SGD` y mira cuanto cambia la convergencia.
